## Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

1. Role - Identifies the message type (e.g. system, user)

2. Content - Represents the actual content of the message (like text, images, audio, documents, etc.)

3. Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [14]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("openai:gpt-4o-mini")




In [2]:
model.invoke("Please tell me what is AI in one word")

AIMessage(content='Intelligence', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebd47-f712-7251-9889-f8d1fc8e3d55-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 1, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

### Text Prompts

Text Prompt are strings - ideal for straight forward generation tasks where you dont need to retain conversation history.

In [3]:
model.invoke("What is langchain?")

AIMessage(content="Langchain is a **framework for developing applications powered by large language models (LLMs)**. In simpler terms, it's a toolkit that helps you build more sophisticated and powerful applications that leverage the capabilities of LLMs like GPT-3, GPT-4, or any other LLM you might want to use.\n\nThink of it this way: LLMs are incredibly powerful language generators, but they are often used in isolation. Langchain provides the **scaffolding and components** to connect LLMs to other data sources, allow them to interact with their environment, and chain together multiple LLM calls and other operations to create complex workflows.\n\nHere's a breakdown of what that means and why it's useful:\n\n**Key Concepts and Components of Langchain:**\n\n*   **LLMs:** Langchain is built around the idea of interacting with LLMs. It provides standardized interfaces to connect to various LLM providers (OpenAI, Hugging Face, Anthropic, etc.).\n*   **Prompts:** Langchain helps you manag

Use text prompts when:

* You have a single, standalone request
* You don't need conversation history
* You want minimal code complexity

### Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

* System message - Tells the model how to behave and provide context for interactions
* Human message - Represents user input and interactions with the model
* AI message - Responses generated by the model, including text content, tool calls, and metadata
* Tool message - Represents the outputs of tool calls

### System Message

A SystemMessage represent an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

### Human Message

A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message

An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message

For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert, and your speciality is writing short poems with a nice ring to it."),
    HumanMessage("Write a poem on moon"),
]

response = model.invoke(messages)

In [7]:
response.content

'A silver disc, a gentle gleam,\nIt sails across the velvet dream.\nA silent watchman, pale and bright,\nIt guides us through the darkest night.'

In [8]:
## Detailed information to LLM through system message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explainations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")

]

response = model.invoke(messages)
print(response.content)


Creating a REST API involves defining a set of rules and conventions for how clients and servers communicate over HTTP. Here's a breakdown of the core concepts and how to implement them using Python, focusing on the popular Flask framework.

## Core Concepts of REST APIs

1.  **Resources:** Everything in a REST API is a resource. This could be a user, a product, an order, etc. Resources are identified by unique URIs (Uniform Resource Identifiers).
2.  **HTTP Methods (Verbs):** Standard HTTP methods are used to perform actions on resources:
    *   `GET`: Retrieve a resource or a collection of resources.
    *   `POST`: Create a new resource.
    *   `PUT`: Update an existing resource (replace the entire resource).
    *   `PATCH`: Partially update an existing resource.
    *   `DELETE`: Remove a resource.
3.  **Representations:** Resources can be represented in various formats, commonly JSON or XML. JSON is widely preferred for its simplicity and efficiency.
4.  **Statelessness:** Each

In [11]:
## Message Metadata

human_msg = HumanMessage(
    content="Hello!",
    name="Harsh", # optional: identify different users
    id = "msg_123", # optional: unique identifier for tracing
)

In [12]:
response = model.invoke([
    human_msg
])
response

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebd58-cbba-7940-961b-007b2ba6db93-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 10, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}})

In [15]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg, # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 equals 4.


In [16]:
response.usage_metadata

{'input_tokens': 47,
 'output_tokens': 8,
 'total_tokens': 55,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [17]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)

ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 32 degree C"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id = "call_123" # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message, # Model's tool call
    tool_message # Tool execution result
]
response = model.invoke(messages) # Model processes the result


In [19]:
tool_message

ToolMessage(content='Sunny, 32 degree C', tool_call_id='call_123')

In [18]:
response

AIMessage(content='The weather in San Francisco is sunny with a temperature of 32°C.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 43, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_323641b728', 'id': 'chatcmpl-Dq2DTBPkuwDHypsUd1eVCjcBxHh0q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebd64-9651-7703-97e7-4df3f614681b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 15, 'total_tokens': 58, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})